# 🦷 DENTEX 4-Class Abnormal Tooth Detection Benchmark (Kaggle 1-Click)

**GitHub Repository:** [https://github.com/tmanhococ/dentex-benchmark](https://github.com/tmanhococ/dentex-benchmark)

Tự động chạy toàn bộ pipeline chỉ với **1 click "Run All"** trên Kaggle GPU (P100 / T4 x2):
1. Tự động clone mã nguồn từ repo GitHub `tmanhococ/dentex-benchmark`.
2. Kiểm tra GPU và cài đặt toàn bộ thư viện cần thiết.
3. Tự động nhận diện dataset DENTEX (từ Kaggle Input hoặc tải từ Hugging Face Hub) và chuẩn hóa nhãn sang COCO 4-class (`Caries`, `Deep Caries`, `Periapical Lesion`, `Impacted Tooth`).
4. Huấn luyện lần lượt 3 kiến trúc: **Faster R-CNN**, **DINO Transformer**, và **HierarchicalDet** (Diffusion baseline).
5. Đánh giá đối soát chuẩn COCO ($mAP_{50}$, $mAP_{50:95}$, AP từng lớp), xuất báo cáo Markdown và bảng mã nguồn LaTeX chuẩn bài báo khoa học.
6. Trực quan hóa kết quả dự đoán bounding box trên ảnh X-quang răng thực tế.

## 1. Clone Mã Nguồn từ GitHub vào Thư Mục Làm Việc Kaggle

In [ ]:
import os
import sys
import shutil
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working/Dentex")
REPO_URL = "https://github.com/tmanhococ/dentex-benchmark.git"

if KAGGLE_WORKING.exists():
    shutil.rmtree(KAGGLE_WORKING)

print(f"🚀 Đang clone mã nguồn từ {REPO_URL}...")
!git clone {REPO_URL} {KAGGLE_WORKING}

%cd {KAGGLE_WORKING}
if str(KAGGLE_WORKING) not in sys.path:
    sys.path.insert(0, str(KAGGLE_WORKING))

print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")

## 2. Kiểm Tra Phần Cứng GPU & Cài Đặt Tự Động Thư Viện

In [ ]:
!pip install --quiet pycocotools albumentations tabulate
!python scripts/setup_env.py

## 3. Chuẩn Bị Dữ Liệu DENTEX (Smart Auto-Detection & COCO 4-Class)

In [ ]:
# Kiểm tra nếu đã gắn dataset DENTEX trên Kaggle Input (VD: rabiayilmaz/dentex-challenge-2023)
kaggle_input = Path("/kaggle/input")
dentex_inputs = list(kaggle_input.glob("*dentex*")) if kaggle_input.exists() else []

if dentex_inputs:
    data_src = dentex_inputs[0]
    print(f"📦 Phát hiện dataset có sẵn tại Kaggle Input: {data_src}")
    !python scripts/download_and_prepare.py --data_dir "{data_src}" --seed 42
else:
    print("🌐 Không tìm thấy Kaggle Input, tự động tải trực tiếp từ Hugging Face Hub (ibrahimhamamci/DENTEX)...")
    !python scripts/download_and_prepare.py --data_dir data --seed 42

## 4. Huấn Luyện 3 Mô Hình (Faster R-CNN, DINO, HierarchicalDet)

In [ ]:
# Huấn luyện 15 epochs cho mỗi kiến trúc trên GPU P100/T4
print("\n==========================================================")
print("🏋️ [1/3] HUẤN LUYỆN FASTER R-CNN (ResNet50-FPN v2)...")
print("==========================================================")
!python scripts/train.py --model faster_rcnn --epochs 15

print("\n==========================================================")
print("🏋️ [2/3] HUẤN LUYỆN DINO (Vision Transformer Detector)...")
print("==========================================================")
!python scripts/train.py --model dino --epochs 15

print("\n==========================================================")
print("🏋️ [3/3] HUẤN LUYỆN HIERARCHICALDET (Diffusion Baseline)...")
print("==========================================================")
!python scripts/train.py --model hierarchical_det --epochs 15

## 5. Đánh Giá Benchmark & Xuất Báo Cáo Khoa Học (Markdown & LaTeX)

In [ ]:
!python scripts/evaluate.py --compare_all

from IPython.display import display, Markdown

summary_file = Path("outputs/benchmark_summary.md")
if summary_file.exists():
    display(Markdown(summary_file.read_text(encoding="utf-8")))

tex_file = Path("outputs/benchmark_table.tex")
if tex_file.exists():
    print("\n==========================================================")
    print("📄 MÃ NGUỒN BẢNG LATEX (Dùng để dán trực tiếp vào Overleaf/Bài báo):")
    print("==========================================================\n")
    print(tex_file.read_text(encoding="utf-8"))

## 6. Trực Quan Hóa Bounding Box Dự Đoán Trên Ảnh X-Quang Thực Tế

In [ ]:
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from dentex_bench.config import BenchmarkConfig
from dentex_bench.models import FasterRCNNRunner

sample_images = list(Path("data").glob("**/*.png")) + list(Path("data").glob("**/*.jpg"))
if sample_images:
    test_img = sample_images[0]
    print(f"🖼️ Hiển thị dự đoán trên ảnh mẫu: {test_img}")
    
    cfg = BenchmarkConfig.from_auto_env()
    runner = FasterRCNNRunner(cfg)
    runner.build_model()
    ckpt = Path("outputs/faster_rcnn/best.pth")
    if ckpt.exists():
        runner.load_checkpoint(ckpt)
        
    bgr = cv2.imread(str(test_img))
    if bgr is not None:
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        preds = runner.predict([tensor])[0]
        
        keep = preds["scores"] >= 0.3
        boxes = preds["boxes"][keep].cpu().numpy()
        labels = preds["labels"][keep].cpu().numpy()
        scores = preds["scores"][keep].cpu().numpy()
        
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(rgb)
        colors = {1: "#E69F00", 2: "#D55E00", 3: "#0072B2", 4: "#009E73"}
        names = {1: "Caries", 2: "Deep Caries", 3: "Periapical", 4: "Impacted"}
        
        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box
            c = colors.get(int(label), "yellow")
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=c, facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, max(0, y1 - 8), f"{names.get(int(label), 'Lesion')} {score:.2f}", color=c, fontsize=10, weight="bold", bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
        plt.axis("off")
        plt.title("DENTEX 4-Class Pathology Predictions (Faster R-CNN)", fontsize=14, weight="bold")
        plt.show()
else:
    print("Chưa có ảnh trong data/. Vui lòng chạy bước 3 để chuẩn bị dữ liệu.")